# H8 — Classical and quantum solver arena
Run the same immutable campaign through exact enumeration and bounded, locally simulated QAOA. The web arena replays these production traces. Playback aligns progress rather than pretending that the algorithms ran concurrently or at the displayed speed.

All examples are synthetic. No cell submits a provider job, requires a secret or spends credits. The H7 hardware section reads the archived result only.

In [1]:
from pathlib import Path
import sys, json
from html import escape
import numpy as np
from IPython.display import HTML, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/app/solver_arena.py").exists())
for path in (ROOT, ROOT / "backend"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from app.solver_arena import ArenaRequest, compare_solvers, load_arena_scenario, recorded_hardware
from app.hybrid_quantum import compile_qubo, compact_model, qaoa_statevector
from shared.hybrid import fingerprint

def table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag}>{escape(str(value))}</{tag}>" for value in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(values, "td") for values in rows) + "</table>"))
print("Production arena imported; no cloud client loaded.")

Production arena imported; no cloud client loaded.


## 1. Exact shared input and explicit execution budget
The recovery fixture is the exact two-qubit campaign submitted in H7. Its terminal-physics outputs were frozen rather than recomputed, because last-bit floating-point differences between platforms can otherwise change its fingerprint. This is a recorded scenario, not live telemetry.

In [2]:
settings = ArenaRequest(preset_id="recovery", depth=2, evaluations=80, shots=512, seed=17)
scenario = load_arena_scenario(settings.preset_id)
before = fingerprint(scenario)
run = compare_solvers(settings)
assert before == run["input_fingerprint"] == "176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3"
assert run["hardware_jobs"] == run["credits_spent"] == 0
assert run["classical"]["result"]["input_fingerprint"] == run["quantum"]["result"]["input_fingerprint"]
print(settings.model_dump())
print("Input identity:", before)
print("Scenario:", scenario.description)
table(["Mission", "Option", "Window UTC", "Metrics"], [(c.mission_id, c.option_id, str(c.window), c.metrics) for c in scenario.candidates])

{'preset_id': 'recovery', 'depth': 2, 'evaluations': 80, 'shots': 512, 'seed': 17}
Input identity: 176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3
Scenario: Synthetic two-mission campaign


Mission,Option,Window UTC,Metrics
follow,follow-early,"start_utc=datetime.datetime(2030, 1, 1, 0, 0, 33, tzinfo=TzInfo(0)) end_utc=datetime.datetime(2030, 1, 1, 0, 0, 34, tzinfo=TzInfo(0))","{'utility': 40.0, 'recovery_propellant_kg': 0.0}"
follow,follow-late,"start_utc=datetime.datetime(2030, 1, 1, 0, 0, 40, tzinfo=TzInfo(0)) end_utc=datetime.datetime(2030, 1, 1, 0, 0, 41, tzinfo=TzInfo(0))","{'utility': 10.0, 'recovery_propellant_kg': 0.0}"
landing,land-backup,"start_utc=datetime.datetime(2030, 1, 1, 0, 0, 29, 999945, tzinfo=TzInfo(0)) end_utc=datetime.datetime(2030, 1, 1, 0, 0, 31, 999945, tzinfo=TzInfo(0))","{'utility': 10.0, 'recovery_propellant_kg': 14.883838625545309}"
landing,land-remote,"start_utc=datetime.datetime(2030, 1, 1, 0, 0, 34, 999980, tzinfo=TzInfo(0)) end_utc=datetime.datetime(2030, 1, 1, 0, 0, 36, 999980, tzinfo=TzInfo(0))","{'utility': 100.0, 'recovery_propellant_kg': 17.736404276589383}"


## 2. Completed-run scoreboard
The classical solver enumerates every assignment; its feasible fraction is a fraction of the search space, not a sampling success probability. Quantum feasible and optimal rates refer to the final sampled shots. Missing plans and gaps remain `None`, never a fabricated zero.

In [3]:
classic, quantum = run["classical"], run["quantum"]
q = quantum["result"]
assert classic["result"]["status"] == "optimal" and quantum["status"] == "completed"
table(["Metric", "Classical", "QAOA local simulation"], [
    ["Best feasible score", classic["score"], quantum["score"]],
    ["Gap to exact optimum", classic["optimum_gap"], quantum["optimum_gap"]],
    ["Solver + validation s", classic["elapsed_s"], quantum["elapsed_s"]],
    ["Feasible measured outcomes", "Not a sampling method", q["sampled_feasible_rate"]],
    ["Optimal measured outcomes", "Not a sampling method", q["sampled_optimal_rate"]],
    ["Selected options", classic["result"]["plan"]["selected_option_ids"], q["best_sampled_plan"]["selected_option_ids"]],
])
print({"preparation_s": run["preparation_s"], "combined_compute_s": run["total_elapsed_s"], "qubits": q["logical_qubits"]})

Metric,Classical,QAOA local simulation
Best feasible score,109.98226359572341,109.98226359572341
Gap to exact optimum,0.0,0.0
Solver + validation s,0.0012412910000421107,0.054294790999847464
Feasible measured outcomes,Not a sampling method,0.98828125
Optimal measured outcomes,Not a sampling method,0.6484375
Selected options,"['follow-late', 'land-remote']","['follow-late', 'land-remote']"


{'preparation_s': 0.0006673340030829422, 'combined_compute_s': 0.056224042004032526, 'qubits': 2}


## 3. Actual classical progress
Each observation follows the real validation of one assignment. It contains the examined plan, cumulative feasible count, best plan and elapsed computation time. These observations do not alter solver selection or its deterministic tie-break.

In [4]:
history = classic["trace"]
assert len(history) == classic["result"]["evaluated_combinations"] == 4
assert history[-1]["best_option_ids"] == tuple(classic["result"]["plan"]["selected_option_ids"])
table(["Examined", "Options", "Feasible", "Best so far", "Elapsed s"], [
    [r["evaluated"], r["option_ids"], r["feasible"], r["best_option_ids"], r["elapsed_s"]] for r in history
])

Examined,Options,Feasible,Best so far,Elapsed s
1,"('follow-early', 'land-backup')",True,"('follow-early', 'land-backup')",0.0008254589993157424
2,"('follow-early', 'land-remote')",False,"('follow-early', 'land-backup')",0.0009347090017399751
3,"('follow-late', 'land-backup')",True,"('follow-early', 'land-backup')",0.000998666997475084
4,"('follow-late', 'land-remote')",True,"('follow-late', 'land-remote')",0.001053833999321796


## 4. Actual QAOA iterations
For every real tuning evaluation, the observer records candidate angles, its ideal distribution, energy and the incumbent distribution. The web checkbox switches between candidate and incumbent views. Histograms show simulator probabilities, while final counts remain separate.

In [5]:
history = quantum["trace"]
model = compact_model(compile_qubo(scenario), max_qubits=2)
assert len(history) == settings.evaluations
for step in history:
    actual = np.abs(qaoa_statevector(model.normalized_energies, step["gammas"], step["betas"])) ** 2
    np.testing.assert_allclose(actual, step["probabilities"], atol=1e-12)
    assert abs(sum(step["best_probabilities"]) - 1) < 1e-12
assert all(b["best_energy"] <= a["best_energy"] for a, b in zip(history, history[1:]))
np.testing.assert_allclose(history[-1]["best_probabilities"], [s["probability"] for s in q["samples"]])
table(["Iteration", "Candidate cost", "Best cost", "Best distribution", "Elapsed s"], [
    [r["iteration"], r["energy"], r["best_energy"], r["best_probabilities"], r["elapsed_s"]] for r in history[::10] + history[-1:]
])

Iteration,Candidate cost,Best cost,Best distribution,Elapsed s
1,5.551115123125783e-17,5.551115123125783e-17,"(0.25, 0.25, 0.25, 0.25)",0.04637412500596838
11,0.08269870971602994,-0.2167272424797365,"(0.45598963557159516, 0.14385867488049175, 0.11233681754073141, 0.2878148720071816)",0.04709595800522948
21,0.24213511366314022,-0.26927135183648304,"(0.5796444174663911, 0.17310324256846332, 0.007413117342966085, 0.23983922262217955)",0.04727633300353773
31,-0.19160861983087157,-0.3398143237422411,"(0.24684289628581546, 0.34806712113383914, 0.0018080886436129071, 0.4032818939367324)",0.04745362500398187
41,-0.4956915782201132,-0.4956915782201132,"(0.3254402772767698, 0.08297473986723031, 0.06436607725846925, 0.5272189055975307)",0.048082708002766594
51,-0.5479316779093634,-0.6496062693935712,"(0.2996935090346603, 0.06187654862684466, 0.00811143826953038, 0.6303185040689647)",0.04827033300534822
61,-0.462553459608637,-0.6496062693935712,"(0.2996935090346603, 0.06187654862684466, 0.00811143826953038, 0.6303185040689647)",0.04844883300393121
71,-0.5607511260227908,-0.6510285426121158,"(0.3095909767879946, 0.05636107146808718, 0.006604622697007818, 0.6274433290469104)",0.048624041002767626
80,-0.6654677399367643,-0.6654677399367643,"(0.29722403040739637, 0.05066976758119427, 0.008436890820646718, 0.6436693111907626)",0.048783291000290774


## 5. Raw outcomes and original constraints
No bitstring is repaired or dropped from the denominator. Every basis state retains its original option IDs, feasibility flag, optimum flag, ideal probability and observed count.

In [6]:
assert sum(s["count"] for s in q["samples"]) == settings.shots
assert quantum["validation"]["status"] == "feasible"
table(["Basis", "Options", "Ideal probability", "Final count", "Feasible", "Optimal"], [
    [s["basis_index"], s["selected_option_ids"], s["probability"], s["count"], s["feasible"], s["optimal"]] for s in q["samples"]
])

Basis,Options,Ideal probability,Final count,Feasible,Optimal
0,"['follow-early', 'land-backup']",0.29722403040739637,155,True,False
1,"['follow-late', 'land-backup']",0.05066976758119427,19,True,False
2,"['follow-early', 'land-remote']",0.008436890820646718,6,False,False
3,"['follow-late', 'land-remote']",0.6436693111907626,332,True,True


## 6. Six-qubit campaigns
These two three-mission cases have four options per mission. The same production API supports independent choices and shared-antenna conflicts, with at most six logical qubits. The exact solver remains the reference.

In [7]:
larger = []
for preset in ("independent", "shared-conflict"):
    item = compare_solvers(ArenaRequest(preset_id=preset, evaluations=16, shots=512, seed=17))
    assert item["quantum"]["result"]["logical_qubits"] == 6
    assert item["classical"]["result"]["total_combinations"] == 64
    larger.append(item)
table(["Preset", "Feasible assignments / 64", "Classical s", "QAOA s", "Observed optimum gap"], [
    [r["request"]["preset_id"], r["classical"]["result"]["feasible_count"], r["classical"]["elapsed_s"], r["quantum"]["elapsed_s"], r["quantum"]["optimum_gap"]] for r in larger
])

Preset,Feasible assignments / 64,Classical s,QAOA s,Observed optimum gap
independent,64,0.0038619170009042136,0.9331947909959126,0.0
shared-conflict,24,0.004715291004686151,0.9903839589969721,0.0


## 7. Infeasible and incomplete inputs
Both panels show why a comparison cannot proceed. Missing capacity is not zero capacity; neither case produces invented quantum results.

In [8]:
blocked = []
for preset, status in (("infeasible", "infeasible"), ("missing-capacity", "incomplete")):
    item = compare_solvers(ArenaRequest(preset_id=preset, evaluations=4))
    assert item["classical"]["result"]["status"] == status
    assert item["quantum"]["status"] == "not_run" and item["quantum"]["result"] is None
    blocked.append(item)
table(["Preset", "Classical status", "QAOA reason"], [[r["request"]["preset_id"], r["classical"]["result"]["status"], r["quantum"]["reason"]] for r in blocked])

Preset,Classical status,QAOA reason
infeasible,infeasible,classical_infeasible
missing-capacity,incomplete,classical_incomplete


## 8. Seed sensitivity and reproducibility
The random seed controls the local training and sampling streams; it does not replace the immutable scenario seed. Timing depends on the machine and load. Repeated seeded distributions and counts should match; durations need not.

In [9]:
repeated = compare_solvers(settings)
strip_time = lambda r: {k: v for k, v in r["quantum"]["result"].items() if k != "elapsed_s"}
assert strip_time(repeated) == strip_time(run)
seed_runs = [run] + [compare_solvers(settings.model_copy(update={"seed": s})) for s in (23, 41)]
table(["Seed", "Feasible shot rate", "Optimal shot rate", "Optimum gap", "QAOA s"], [
    [r["request"]["seed"], r["quantum"]["result"]["sampled_feasible_rate"], r["quantum"]["result"]["sampled_optimal_rate"], r["quantum"]["optimum_gap"], r["quantum"]["elapsed_s"]] for r in seed_runs
])

Seed,Feasible shot rate,Optimal shot rate,Optimum gap,QAOA s
17,0.98828125,0.6484375,0.0,0.054294790999847464
23,0.9609375,0.77734375,0.0,0.03397216699522687
41,0.99609375,0.95703125,0.0,0.03442133299540728


## 9. Archived real-device evidence
This reads the single completed H7 Rigetti job, checks its input fingerprint, and validates raw counts against the original constraints. Its 160-evaluation / 256-shot budget differs from this notebook's new local runs. The recorded 118.795 seconds includes network, queue and polling, not isolated device execution time. There is no new cloud submission.

In [10]:
hardware = recorded_hardware()
assert hardware["input_fingerprint"] == run["input_fingerprint"]
assert hardware["new_jobs_submitted"] == 0
assert hardware["sampled_optimal_rate"] == 200 / 256
assert hardware["sampled_feasible_rate"] == 247 / 256
assert hardware["optimum_gap"] == 0
print(json.dumps(hardware, indent=2))

{
  "execution_mode": "hardware",
  "job_id": "4c0adc79-6bd3-4578-9147-bbf8ab7b2209",
  "backend_code": "rigetti:cepheus-1-108q",
  "input_fingerprint": "176fe28ceab9f98edef3ddc8d64175e0574d3459de96242a3705275cfbdf6bf3",
  "circuit_sha256": "54275a0b2994acac9eb46f152aa9215d9ce12be4c922e24c4022dbb49c72298d",
  "shots": 256,
  "counts": {
    "00": 18,
    "01": 29,
    "10": 9,
    "11": 200
  },
  "logical_qubits": 2,
  "depth": 2,
  "training_seed": 17,
  "training_evaluations": 160,
  "credits_spent": 1,
  "credit_type": "Spark",
  "submitted_utc": "2026-11-14T20:10:08.326044Z",
  "completion_observed_utc": "2026-11-14T20:12:07.120647Z",
  "observed_elapsed_s": 118.794603,
  "provenance": "docs/openquantum-hardware.md",
  "attribution_url": "https://www.openquantum.com/citation",
  "limitation": "One archived trial; different budgets from new runs. No hardware speedup or calibrated noise claim.",
  "kind": "recorded_hardware",
  "new_jobs_submitted": 0,
  "samples": [
    {
      "ba

## 10. Interpretation and API entry points
Classical enumeration already solves these examples exactly. QAOA uses classical preprocessing, exhaustive mapping audits and local parameter tuning, and its measured time includes that work. Higher probability than uniform sampling is not quantum advantage. This is an explanatory comparison, not a large-scale performance study.

The web view is available at `http://localhost:5174/#/solver-arena`. Backend routes are `GET /arena/presets`, `POST /arena/compare` and `GET /arena/hardware`. A single process admits one arena operation at a time; overlapping calls receive 429. Fixed presets and strict budget limits bound work. The browser can cancel waiting, but the server finishes its bounded computation. The exported JSON preserves inputs, settings, raw outcomes and traces.

Infrastructure attribution for the archived record: [Open Quantum citation guidance](https://www.openquantum.com/citation), Kuhn, Armbruster and Wold (2026), [doi:10.5281/zenodo.22103261](https://doi.org/10.5281/zenodo.22103261). See `docs/solver-arena.md` for assumptions and verification.

In [11]:
summary = {"mode": run["execution_mode"], "new_provider_jobs": run["hardware_jobs"], "credits_spent_this_notebook": 0,
           "classical_status": classic["result"]["status"], "quantum_status": quantum["status"],
           "classical_trace_steps": len(classic["trace"]), "quantum_trace_steps": len(quantum["trace"]),
           "recorded_h7_job": hardware["job_id"]}
print(json.dumps(summary, indent=2))
print("H8 assertions passed. No provider connection or credit use.")

{
  "mode": "noiseless_simulator",
  "new_provider_jobs": 0,
  "credits_spent_this_notebook": 0,
  "classical_status": "optimal",
  "quantum_status": "completed",
  "classical_trace_steps": 4,
  "quantum_trace_steps": 80,
  "recorded_h7_job": "4c0adc79-6bd3-4578-9147-bbf8ab7b2209"
}
H8 assertions passed. No provider connection or credit use.
